In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os

# Get Code Encoding


In [3]:
os.environ["http_proxy"] = "http://127.0.0.1:11452"
os.environ["https_proxy"] = "http://127.0.0.1:11452"

In [4]:
# Load model directly
from transformers import RobertaTokenizer, RobertaModel
from model import Model

MODEL_NAME = "microsoft/graphcodebert-base"
MODEL_PATH = "/data0/mlcask/scs/CodeBERT/GraphCodeBERT/Siamese-model/demo/python_model"

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaModel.from_pretrained(MODEL_PATH)
model = Model(model)

In [5]:
from dataclasses import asdict
from dfg import CodeEncodingBuilder

In [6]:
code_str = """# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.comodel = Model(model)m/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
for filename in filenames:
print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.
"""

In [7]:
code_str = """import time
for i in range(15):
    print(i)
    time.sleep(1)"
"""

In [8]:
encoding_builder = CodeEncodingBuilder(tokenizer, model)
encoding_builder

<CodeEncodingBuilder device=cuda:1>

In [9]:
f = encoding_builder.make_input_features(code_str)
f

code_tokens (31): ['<s>', 'import', '_time', '_for', '_i', '_in', '_range', '_(', '_15', '_)', '_:', '_print', '_(', '_i', '_)', '_time', '_.', '_sleep', '_(', '_1', '_)', '_"', '</s>', 'time', 'i', 'range', '15', 'print', 'i', 'time', 'sleep']
code_ids (320): 0 41975 86 13 939 11 1186 36 379 4839 4832 5780 36 939 4839 86 479 3581 36 112 4839 22 2 3 3 3 3 3 3 3 3 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
position_idx (320): 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16

In [10]:
input = encoding_builder.make_input(code_str)
input[0].shape, input[1].shape, input[2].shape

(torch.Size([320]), torch.Size([320, 320]), torch.Size([320]))

In [12]:
output = encoding_builder.get_encoding(input)
print(output.shape)
print(output[:, :16])

torch.Size([1, 768])
tensor([[-0.4046, -0.1858, -0.3654, -0.0687,  0.1600,  0.1680,  0.1658,  0.0766,
         -0.5253, -0.4134,  0.6512, -0.6090, -0.4404, -0.2613, -0.3834,  0.5255]],
       device='cuda:1', grad_fn=<SliceBackward0>)


# Preprocessing


In [43]:
from nb2p import database
from nb2p.notebook import Notebook
from dfg import get_code_encoding
from dfgtree import preprocess, DFGTree
from nb2p.astparse import parser as get_parser

parser, lang = get_parser()

In [14]:
db, client = database.connect(dataset_name="distilkaggle", verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


In [15]:
nb_data = database.get_notebooks(
    db, {"prompted": True, "segments.6": {"$exists": True}}, include_segments=True
).next()
nb = Notebook.from_db_result(nb_data)
print(nb.id)
# nb.pretty_print_segments()

66ee62b3a6a40d28d333d09c


In [61]:
preproc_result = preprocess(nb, parser, lang)
preproc_result

{'imports': ['import numpy as np',
  'import matplotlib.pyplot as plt',
  'from sklearn.metrics import accuracy_score',
  'import sklearn',
  'import sklearn.datasets',
  'import warnings',
  'import sklearn.linear_model',
  'import sklearn.tree',
  'import sklearn.svm',
  'from sklearn.linear_model import LinearRegression',
  'from sklearn.metrics import r2_score',
  'from sklearn.metrics import mean_squared_error',
  'from sklearn.preprocessing import PolynomialFeatures',
  'from sklearn.pipeline import Pipeline',
  'from sklearn import cross_validation',
  'import timeit'],
 'func_defs': [('jitter',
   'def jitter(X, scale):\n    if scale > 0:        \n        return X + np.random.normal(0, scale, X.shape)\n    return X'),
  ('jitter_test',
   'def jitter_test(classifier, X, y, metric_FUNC = accuracy_score, sigmas = np.linspace(0, 0.5, 30), averaging_N = 5):\n    out = []\n    for s in sigmas:\n        averageAccuracy = 0.0\n        for x in range(averaging_N):\n            averageA

In [62]:
for func_name, func_code in preproc_result["func_defs"]:
    get_code_encoding(func_code, encoding_builder)

code_tokens (42): ['<s>', 'def', '_j', 'itter', '_(', '_X', '_,', '_scale', '_)', '_:', '_if', '_scale', '_>', '_0', '_:', '_return', '_X', '_+', '_np', '_.', '_random', '_.', '_normal', '_(', '_0', '_,', '_scale', '_,', '_X', '_.', '_shape', '_)', '_return', '_X', '</s>', 'X', 'scale', 'scale', 'X', 'scale', 'X', 'X']
code_ids (320): 0 9232 1236 7915 36 1577 2156 3189 4839 4832 114 3189 8061 321 4832 671 1577 2055 46446 479 9624 479 2340 36 321 2156 3189 2156 1577 479 3989 4839 671 1577 2 3 3 3 3 3 3 3 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [65]:
DFGTree(input=preproc_result, builder=encoding_builder).build()

{'func_defs': [{'name': 'jitter',
   'code': 'def jitter(X, scale):\n    if scale > 0:        \n        return X + np.random.normal(0, scale, X.shape)\n    return X',
   'repr': <DFGNode repr=(1, 768) children=[])>},
  {'name': 'jitter_test',
   'code': 'def jitter_test(classifier, X, y, metric_FUNC = accuracy_score, sigmas = np.linspace(0, 0.5, 30), averaging_N = 5):\n    out = []\n    for s in sigmas:\n        averageAccuracy = 0.0\n        for x in range(averaging_N):\n            averageAccuracy += metric_FUNC( y, classifier.predict(jitter(X, s)))\n        out.append( averageAccuracy/averaging_N)\n    return (out, sigmas, np.trapz(out, sigmas))',
   'repr': <DFGNode repr=(1, 768) children=[])>},
  {'name': 'plotter',
   'code': "def plotter(model, X, Y, ax, npts=5000):\n    xs = []\n    ys = []\n    cs = []\n    for _ in range(npts):\n        x0spr = max(X[:,0])-min(X[:,0])\n        x1spr = max(X[:,1])-min(X[:,1])\n        x = np.random.rand()*x0spr + min(X[:,0])\n        y = np.ra